# **Import Packages**

In [1]:
import os
import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm
from PIL import Image, ImageDraw, ImageFont

from ultralytics import YOLO
import torch

import easyocr
import cv2
import re
import os
import glob

In [2]:
from ultralytics import __version__ as yolo_version

print(f"Ultralytics YOLO version: {yolo_version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Pillow version: {Image.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"Numpy version: {np.__version__}")

Ultralytics YOLO version: 8.3.56
PyTorch version: 2.4.1+cu118
Pillow version: 10.4.0
OpenCV version: 4.10.0
Numpy version: 1.24.4


# **EasyOCR**

In [ ]:
# 명판 검사 함수
def EasyOCR(img_path, reader):

    # 1. 이미지 정보
    image = cv2.imread(img_path)      # 이미지 불러오기
    height, width = image.shape[:2]   # 이미지 크기 확인

    # 2. 명판 위치만 크롭
    x_min, y_min, x_max, y_max = (0, 0, int(width/2), int(height-1))  # 이미지의 왼쪽에 이미지가 있기때문에 이미지 절반만 크롭
    cropped_img = image[y_min:y_max, x_min:x_max]

    # 3. 명판이 돌려져 있기에 똑바로 돌리기
    cropped_width, cropped_height = int(width/2), int(height-1)       
    rotation_matrix = cv2.getRotationMatrix2D(center = (cropped_width//2, cropped_height//2), angle = -90, scale = 1.0)
    rotated_img = cv2.warpAffine(cropped_img, rotation_matrix, (cropped_width, cropped_height))


    # 4. 전처리된 이미지에서 문자 읽기
    read_texts = reader.readtext(rotated_img)        # 문자 읽기
    label_texts = [text[1] for text in read_texts]   # 필요한 결과(문자 정보)만 선택
    label_texts[1] = label_texts[1][:-5].replace("O", "0") + label_texts[1][-5:]

    # 5. 이미지 파일명에서 필요한 부분 추출
    filename_info = re.findall(r"\((.*?)\)", img_path)[0].split(',')    # 괄호()안에 있는 문자만 선택

    # 6. 비교 수행
    result = "Error"
    try:
        # 비교 규칙 생성
        rule1 = label_texts[1][:-5] in filename_info
        rule2 = str(int(label_texts[1][-4:-1])) in filename_info

        # 비교 규칙이 모두 맞으면 No Detection, 하나라도 틀리면 Label Problem
        if rule1 and rule2:
            result = "No Detection"
        else:
            result = "Label problem"
    except:
        pass

    # 7. 문자 정보 저장(파일명, 이미지)
    real_img_text = (" ".join(filename_info),  label_texts[0] + " " + label_texts[1][:-1])

    return result, real_img_text

# **Prediction**

1. 저장된 모델 불러오기
2. 명판 검사(EasyOCR) 수행여부 선택
3. 이미지 예측
4. 결과 출력(Text, Image)

In [3]:
# 1. 저장된 모델 불러오기
pretrained_model_path = "./best.pt"  # 모델 저장 경로
pretrained_model = YOLO(pretrained_model_path)                        # 저장된 모델 불러기기

# 2. 명판 검사 여부
nameplate_check = False

if nameplate_check:
    print("명판 검사를 시작합니다.")
    Reader = easyocr.Reader(["en", "ko"])
else:
    print("명판 검사는 진행하지 않습니다.")

In [2]:
# 3. 이미지 예측

class_names = [name for name in pretrained_model.names.values()]      # 결함 종류(클래스) 가져오기
results_list = []                                                     # 결과 저장 리스트
image_folder_path = "./predict_images"                                # 예측할 이미지가 담긴 폴더
output_folder_path = "./predicted_results"                            # 예측 결과를 저장할 폴더

# input folder내 모든 데이터에 하나씩 접근하며 수행
for image_file in tqdm(os.listdir(input_folder_path)):
  
    # 1. 이미지 파일이 아닌 경우 스킵
    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
        
    # 2. 이미지 경로 설정
    image_path = os.path.join(input_folder_path, image_file)
    
    # 3. 명판 분석
    if nameplate_check:
        name_result, real_img_text = EasyOCR(img_path = image_path, reader = Reader)

    # 4. 이미지를 PIL로 로드
    image = Image.open(image_path).convert("RGB")

    # 5. YOLO 모델을 통한 결함 예측
    results = pretrained_model.predict(source = image_path, conf=0.5)

    # 6. 결과 처리
    ## 6-1. 이미지(매 반복마다 저장)
    ### 6-1-1. 예측 결과가 있으면, (위치, Confidence score, 결함 예측 클래스) 정보를 저장
    if len(results[0].boxes.data.cpu().numpy()) > 0: 
        x_min, y_min, x_max, y_max, confidence, predicted_class = results[0].boxes.data.cpu().numpy()[0]
        label = class_names[int(predicted_class)] # 결함 클래스 (숫자 -> 문자)

    ### 6-1-2. 예측 결과 없으면, 위치와 Confidence Score는 0, 결함 예측 결과는 No Detection 
    else:
        x_min, y_min, x_max, y_max = 0, 0, 0, 0
        confidence = ""
        label = "No Detection"
        
    ### 6-1-3. 이미지에 예측값 표시
    draw = ImageDraw.Draw(image)
    draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
    font =  ImageFont.truetype("arial.ttf", 40)
    draw.text((10, 10), "Prediction: " + label, fill="red", font = font)
    
    if nameplate_check:
        draw.text((10, 90), "Nameplate: " + name_result, fill="red", font = font)
        draw.text((10, 170), "True info: " + real_img_text[0], fill="#00FFFF", font = font)
        draw.text((10, 250), "Observed info: " + real_img_text[1], fill="#00FFFF", font = font)

    ### 6-1-4. 결과 이미지를 출력 폴더에 저장
    output_image_path = os.path.join(output_folder_path, f"pred_{image_file}")
    image.save(output_image_path)
        

    ## 6-2. 데이터프레임(반복문 완료 후 한번에 저장)
    if nameplate_check:
        results_list.append({"Image Name": image_file, "Prediction": label, "Confidence" : confidence,
                             "Nameplate": name_result, "Text(true, read)" : real_img_text})
    else:
        results_list.append({"Image Name": image_file, "Prediction": label, "Confidence" : confidence})

In [ ]:
# 결과 데이터프레임 생성 및 저장
yolo_results_df = pd.DataFrame(results_list)

output_csv_path = os.path.join(output_folder_path, "Yolo_predictions.csv")
yolo_results_df.to_csv(output_csv_path, index = False)

print("최종 결과 저장 완료!")